In [2]:
"""Membro 3 - Seleção da linha e previsão.

Entrada de cada candidata (vinda do Membro 2):
  {"codigo_linha": 111, "letreiro": "8000-10", "sentido": 1,
   "paradas": [{"cp": 1, "np": "Nome", "py": -23.55, "px": -46.63}, ...]}
"""
from math import radians, sin, cos, asin, sqrt

BASE_URL = "http://api.olhovivo.sptrans.com.br/v2.1"  # alinhar com o Membro 1


def haversine(p1, p2):
    """Distância em metros entre (lat, lon) e (lat, lon). Pode usar a do Membro 2."""
    dlat, dlon = radians(p2[0] - p1[0]), radians(p2[1] - p1[1])
    a = sin(dlat / 2) ** 2 + cos(radians(p1[0])) * cos(radians(p2[0])) * sin(dlon / 2) ** 2
    return 2 * 6371000 * asin(sqrt(a))


def parada_mais_proxima(ponto, paradas):
    """Devolve (índice, distância) da parada mais perto do ponto."""
    dists = [haversine(ponto, (p["py"], p["px"])) for p in paradas]
    i = dists.index(min(dists))
    return i, dists[i]


def escolher_linha(candidatas, origem, destino, raio):
    """Escolhe a linha com a menor caminhada (origem->embarque + desembarque->destino)."""
    try:
        melhor = None
        for linha in candidatas:
            paradas = linha["paradas"]
            i, d1 = parada_mais_proxima(origem, paradas)    # embarque
            j, d2 = parada_mais_proxima(destino, paradas)   # desembarque
            if d1 > raio or d2 > raio or i >= j:            # fora do raio ou ordem errada
                continue
            if melhor is None or d1 + d2 < melhor["caminhada_total"]:
                melhor = {
                    "codigo_linha": linha["codigo_linha"],
                    "letreiro": linha.get("letreiro"),
                    "sentido": linha.get("sentido"),
                    "embarque": paradas[i],
                    "desembarque": paradas[j],
                    "intermediarias": paradas[i + 1:j],
                    "dist_origem_embarque": round(d1),
                    "dist_desembarque_destino": round(d2),
                    "caminhada_total": round(d1 + d2),
                }
        if melhor is None:
            return {"ok": False, "erro": "Nenhuma linha atende origem e destino no raio e na ordem do trajeto."}
        return {"ok": True, **melhor}
    except Exception as e:
        return {"ok": False, "erro": f"Erro ao escolher a linha: {e}"}


def consultar_previsao(session, codigo_parada, codigo_linha):
    """Próximo ônibus no ponto de embarque. Se não houver, devolve um aviso."""
    try:
        r = session.get(f"{BASE_URL}/Previsao",
                        params={"codigoParada": codigo_parada, "codigoLinha": codigo_linha},
                        timeout=10)
        r.raise_for_status()
        dados = r.json()
        horarios = [v["t"] for l in (dados.get("p") or {}).get("l", [])
                    if l.get("cl") == codigo_linha for v in l.get("vs", [])]
        if not horarios:
            return {"disponivel": False, "aviso": "Previsão indisponível: nenhum ônibus previsto agora."}
        return {"disponivel": True, "proximo": min(horarios)}
    except Exception as e:
        return {"disponivel": False, "aviso": f"Previsão indisponível: falha ao consultar a API ({e})."}


def selecionar_e_prever(session, candidatas, origem, destino, raio):
    """Usar no notebook: escolhe a linha e já traz a previsão."""
    resultado = escolher_linha(candidatas, origem, destino, raio)
    if resultado["ok"]:
        resultado["previsao"] = consultar_previsao(session, resultado["embarque"]["cp"], resultado["codigo_linha"])
    return resultado


if __name__ == "__main__":  # teste sem token
    class Resp:
        def __init__(self, d): self.d = d
        def raise_for_status(self): pass
        def json(self): return self.d
    class Sessao:
        def __init__(self, d): self.d = d
        def get(self, *a, **k): return Resp(self.d)

    cand = [{"codigo_linha": 111, "letreiro": "TESTE-10", "sentido": 1, "paradas": [
        {"cp": 1, "np": "A", "py": -23.5502, "px": -46.6302},
        {"cp": 2, "np": "B", "py": -23.5550, "px": -46.6400},
        {"cp": 3, "np": "C", "py": -23.5598, "px": -46.6498}]}]
    o, d = (-23.5500, -46.6300), (-23.5600, -46.6500)
    prev = {"p": {"l": [{"cl": 111, "vs": [{"t": "10:05"}, {"t": "10:12"}]}]}}
    print(selecionar_e_prever(Sessao(prev), cand, o, d, 500))
    print(selecionar_e_prever(Sessao({"p": None}), cand, o, d, 500)["previsao"])
    print(selecionar_e_prever(Sessao(prev), cand, o, d, 5))

{'ok': True, 'codigo_linha': 111, 'letreiro': 'TESTE-10', 'sentido': 1, 'embarque': {'cp': 1, 'np': 'A', 'py': -23.5502, 'px': -46.6302}, 'desembarque': {'cp': 3, 'np': 'C', 'py': -23.5598, 'px': -46.6498}, 'intermediarias': [{'cp': 2, 'np': 'B', 'py': -23.555, 'px': -46.64}], 'dist_origem_embarque': 30, 'dist_desembarque_destino': 30, 'caminhada_total': 60, 'previsao': {'disponivel': True, 'proximo': '10:05'}}
{'disponivel': False, 'aviso': 'Previsão indisponível: nenhum ônibus previsto agora.'}
{'ok': False, 'erro': 'Nenhuma linha atende origem e destino no raio e na ordem do trajeto.'}
